# Sales Effectiveness — Lead Quality Prediction
### DataMites™ Project Mentoring — Project Ref: PM-PR-0019

**Client:** FicZon Inc  
**Category:** Product Sales  

---

## Business Case

FicZon Inc is an IT solution provider offering products ranging from on-premises solutions to SaaS-based
solutions. FicZon's major lead generation channel is digital, primarily through their website.

FicZon's business is heavily dependent on sales force effectiveness. As the market matures and new
competitors enter, FicZon is experiencing a dip in sales. Effective sales depends on lead quality, which is
currently based on **manual categorization** and highly dependent on sales staff judgement. While a quality
process exists to continuously update lead categorization, its value is limited to post-analysis rather than
real-time conversion support.

FicZon wants to explore Machine Learning to **pre-categorize lead quality**, with the goal of significantly
increasing sales effectiveness.

## Project Goals
1. **Data exploration insights** — understand sales effectiveness drivers.
2. **ML model** to predict the **Lead Category** (`High Potential` / `Low Potential`).


## 1. Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import pymysql

import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")
%matplotlib inline

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    roc_auc_score, roc_curve
)

import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)


## 2. Load Dataset

The dataset is stored in a MySQL database. We connect using `pymysql` and load the `data` table from the
`project_sales` database into a pandas DataFrame.

In [ ]:
conn = pymysql.connect(
    host='18.136.157.135',
    # port=3306,
    user='dm_team2',
    password='DM!$Team&27@9!20!',
    database='project_sales'
)

df = pd.read_sql("SELECT * FROM data", conn)
print("Dataset loaded successfully.")


In [ ]:
# Always close the connection once the data has been pulled
conn.close()


## 3. First Look at the Data

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe(include='all').T

## 4. Missing Value Analysis

From the feature summary in the project brief, we already expect missing values in `Product_ID`, `Source`,
`Mobile`, `Sales_Agent`, and `Location`. Let's confirm.

In [ ]:
missing = df.isnull().sum().to_frame("missing_count")
missing["missing_pct"] = (missing["missing_count"] / len(df) * 100).round(2)
missing.sort_values("missing_count", ascending=False)


In [ ]:
plt.figure(figsize=(8, 4))
sns.heatmap(df.isnull(), cbar=False, cmap="viridis", yticklabels=False)
plt.title("Missing Value Map")
plt.show()


## 5. Target Variable — Lead Category

The raw data does not contain a ready-made "Lead Category" column — it is derived from `Status`. First,
inspect the unique values in `Status` to design the mapping to `High Potential` / `Low Potential`.

In [ ]:
df['Status'].value_counts(dropna=False)


**Note:** Adjust the lists below (`high_potential_status`) after inspecting the actual `Status` values printed
above — group statuses that indicate genuine sales interest / conversion under *High Potential*, and the rest
(junk, not interested, no response, etc.) under *Low Potential*.

In [ ]:
# TODO: update these lists to match the exact Status strings found in your data
high_potential_status = [
    'WON', 'CONVERTED', 'IN PROGRESS', 'INTERESTED', 'OPEN'
]

df['Lead_Category'] = np.where(
    df['Status'].str.upper().isin([s.upper() for s in high_potential_status]),
    'High Potential',
    'Low Potential'
)

df['Lead_Category'].value_counts()


In [ ]:
plt.figure(figsize=(5, 4))
sns.countplot(x='Lead_Category', data=df, palette='Set2')
plt.title('Lead Category Distribution')
plt.show()


## 6. Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(10, 5))
df['Source'].value_counts().plot(kind='bar', color='steelblue')
plt.title('Lead Source Distribution')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10, 5))
sns.countplot(y='Source', hue='Lead_Category', data=df,
              order=df['Source'].value_counts().index)
plt.title('Lead Category by Source')
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10, 5))
top_locations = df['Location'].value_counts().nlargest(15).index
sns.countplot(y='Location', hue='Lead_Category', data=df[df['Location'].isin(top_locations)],
              order=top_locations)
plt.title('Lead Category by Top 15 Locations')
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(x='Delivery_Mode', hue='Lead_Category', data=df)
plt.title('Lead Category by Delivery Mode')
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10, 5))
top_agents = df['Sales_Agent'].value_counts().nlargest(15).index
sns.countplot(y='Sales_Agent', hue='Lead_Category', data=df[df['Sales_Agent'].isin(top_agents)],
              order=top_agents)
plt.title('Lead Category by Top 15 Sales Agents')
plt.tight_layout()
plt.show()


In [ ]:
df['Created'] = pd.to_datetime(df['Created'], errors='coerce')
df['Created_Month'] = df['Created'].dt.to_period('M').astype(str)

plt.figure(figsize=(12, 5))
df.groupby('Created_Month').size().plot(kind='line', marker='o')
plt.title('Leads Created Over Time')
plt.ylabel('Number of Leads')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 7. Data Preprocessing

- Drop identifier-like / high-cardinality free-text columns that don't generalize (`EMAIL`, `Mobile`, `Created`).
- Impute missing values in categorical columns.
- Encode categorical features.
- Encode the target variable.

In [ ]:
model_df = df.drop(columns=['EMAIL', 'Mobile', 'Created', 'Created_Month'], errors='ignore')

# Impute missing categorical values with a placeholder
categorical_cols = ['Source', 'Location', 'Sales_Agent', 'Delivery_Mode']
for col in categorical_cols:
    if col in model_df.columns:
        model_df[col] = model_df[col].fillna('Unknown')

# Impute Product_ID (numeric) with median
if 'Product_ID' in model_df.columns:
    model_df['Product_ID'] = model_df['Product_ID'].fillna(model_df['Product_ID'].median())

model_df.isnull().sum()


In [ ]:
encoders = {}
for col in ['Source', 'Location', 'Sales_Agent', 'Delivery_Mode']:
    if col in model_df.columns:
        le = LabelEncoder()
        model_df[col] = le.fit_transform(model_df[col].astype(str))
        encoders[col] = le

target_le = LabelEncoder()
model_df['Lead_Category_Encoded'] = target_le.fit_transform(model_df['Lead_Category'])

model_df = model_df.drop(columns=['Status', 'Lead_Category'])
model_df.head()


## 8. Train-Test Split

In [ ]:
X = model_df.drop(columns=['Lead_Category_Encoded'])
y = model_df['Lead_Category_Encoded']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)


In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


## 9. Model Building

We train and compare several classification algorithms to predict `Lead_Category`.

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'SVM': SVC(probability=True, random_state=42)
}

results = {}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    preds = model.predict(X_test_scaled)
    acc = accuracy_score(y_test, preds)
    results[name] = acc
    print(f"{name}: Accuracy = {acc:.4f}")


In [ ]:
results_df = pd.DataFrame(list(results.items()), columns=['Model', 'Accuracy']).sort_values(
    'Accuracy', ascending=False
).reset_index(drop=True)
results_df


In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(x='Accuracy', y='Model', data=results_df, palette='viridis')
plt.title('Model Comparison — Accuracy')
plt.xlim(0, 1)
plt.tight_layout()
plt.show()


## 10. Detailed Evaluation of the Best Model

Pick the best-performing model from the comparison above and inspect it in detail
(confusion matrix, classification report, ROC curve, feature importance).

In [ ]:
best_model_name = results_df.iloc[0]['Model']
best_model = models[best_model_name]
print("Best model:", best_model_name)

y_pred = best_model.predict(X_test_scaled)

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, target_names=target_le.classes_))


In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=target_le.classes_, yticklabels=target_le.classes_)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title(f'Confusion Matrix — {best_model_name}')
plt.show()


In [ ]:
if hasattr(best_model, "predict_proba"):
    y_proba = best_model.predict_proba(X_test_scaled)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)

    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, label=f'AUC = {auc:.3f}')
    plt.plot([0, 1], [0, 1], linestyle='--', color='grey')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'ROC Curve — {best_model_name}')
    plt.legend()
    plt.show()


In [ ]:
if hasattr(best_model, "feature_importances_"):
    importances = pd.Series(best_model.feature_importances_, index=X.columns).sort_values(ascending=False)
    plt.figure(figsize=(8, 5))
    sns.barplot(x=importances.values, y=importances.index, palette='mako')
    plt.title(f'Feature Importance — {best_model_name}')
    plt.tight_layout()
    plt.show()
else:
    print(f"{best_model_name} does not expose feature_importances_.")


## 11. Hyperparameter Tuning (Optional Improvement)

A quick `GridSearchCV` example on Random Forest to try to squeeze out extra performance.

In [ ]:
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 5, 10, 20],
    'min_samples_split': [2, 5, 10]
}

grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid.fit(X_train_scaled, y_train)

print("Best params:", grid.best_params_)
print("Best CV accuracy:", grid.best_score_)

tuned_preds = grid.best_estimator_.predict(X_test_scaled)
print("Test accuracy:", accuracy_score(y_test, tuned_preds))


## 12. Conclusion & Business Recommendation

- The notebook loads FicZon's lead data directly from the `project_sales` MySQL database.
- Missing values were found in `Product_ID`, `Source`, `Mobile`, `Sales_Agent`, and `Location`, and were
  imputed appropriately.
- `Lead_Category` (`High Potential` / `Low Potential`) was derived from the `Status` field — **review and
  refine the status grouping in Section 5 against the actual business definition used by FicZon's sales
  team** before finalizing.
- Several classification models were compared; the best-performing model (see Section 9/10) can be used to
  **pre-score incoming leads automatically**, so the sales team can prioritize high-potential leads instead
  of relying purely on manual triage.
- `Source`, `Location`, and `Sales_Agent` are visibly associated with lead quality — this can also inform
  where FicZon invests its marketing spend.

**Next steps:** validate the Status → Lead Category mapping with FicZon's sales team, retrain on a larger /
more recent data pull, and deploy the chosen model as a scoring step in the lead-intake pipeline.
